In [8]:
# Load the raw dataset and find rows containing the Excel error value '#NUM!'
import pandas as pd

raw_df = pd.read_csv("dataset/waterQuality1.csv")
bad_rows = (raw_df.astype(str) == "#NUM!").any(axis=1)

print(f"Shape: {raw_df.shape}")
print(f"Rows containing '#NUM!': {bad_rows.sum()}")
raw_df[bad_rows]

Shape: (7999, 21)
Rows containing '#NUM!': 3


,aluminium,ammonia,arsenic,barium,cadmium,chloramine,chromium,copper,flouride,bacteria,...,lead,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium,is_safe
7551,0.03,#NUM!,0.08,0.79,0.07,0.08,0.05,0.58,0.34,0.00,...,0.183,4.37,1.43,0.007,0.62,2.54,0.07,0.05,0.05,#NUM!
7568,0.06,#NUM!,0.07,1.72,0.08,0.32,0.01,1.11,0.61,0.00,...,0.178,12.10,2.03,0.008,1.37,2.05,0.06,0.10,0.07,#NUM!
7890,0.01,#NUM!,0.08,0.49,0.00,0.07,0.09,0.06,0.72,0.57,...,0.088,9.57,1.45,0.009,7.67,7.70,0.03,0.05,0.02,#NUM!


In [9]:
# Drop the corrupted rows (the target itself is '#NUM!' there) and convert all columns to numbers
df = raw_df[~bad_rows].apply(pd.to_numeric).reset_index(drop=True)
df["is_safe"] = df["is_safe"].astype(int)

print(f"Rows after cleaning: {len(df)}")

Rows after cleaning: 7996


In [10]:
# Verify no missing values remain and check the class balance (0 = not safe, 1 = safe)
print("Missing values:", df.isna().sum().sum())
df["is_safe"].value_counts()

Missing values: 0


is_safe
0    7084
1     912
Name: count, dtype: int64

In [11]:
# Stratified 80/20 train/test split so both sets keep the same safe/unsafe ratio (scaling is done later inside each model's Pipeline)
from sklearn.model_selection import train_test_split

X = df.drop(columns="is_safe")
y = df["is_safe"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
pd.DataFrame({"train %": y_train.value_counts(normalize=True), "test %": y_test.value_counts(normalize=True)}).mul(100).round(2)

Train: (6396, 20) | Test: (1600, 20)


,train %,test %
is_safe,,
0,88.59,88.62
1,11.41,11.38


In [12]:
# Shared imports and a helper that computes the 6 required metrics; test results of every model are collected in `results`
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

def evaluate(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    return {"Accuracy": accuracy_score(y, y_pred),
            "AUC": roc_auc_score(y, y_prob),
            "Precision": precision_score(y, y_pred),
            "Recall": recall_score(y, y_pred),
            "F1": f1_score(y, y_pred),
            "MCC": matthews_corrcoef(y, y_pred)}

results = {}

In [13]:
# Logistic Regression: tune C and class_weight with 5-fold stratified CV (scored on F1), then compare train vs test metrics to check overfitting
from sklearn.linear_model import LogisticRegression

lr_pipe = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=1000))])

lr_grid = GridSearchCV(lr_pipe,
                       param_grid={"model__C": [0.01, 0.1, 1, 10, 100],
                                   "model__class_weight": [None, "balanced"]},
                       scoring="f1", cv=5, return_train_score=True, n_jobs=-1)
lr_grid.fit(X_train, y_train)

best_lr = lr_grid.best_estimator_
results["Logistic Regression"] = evaluate(best_lr, X_test, y_test)

print("Best params:", lr_grid.best_params_, "| CV F1:", round(lr_grid.best_score_, 4))
pd.DataFrame({"Train": evaluate(best_lr, X_train, y_train), "Test": results["Logistic Regression"]}).round(4)

Best params: {'model__C': 0.01, 'model__class_weight': 'balanced'} | CV F1: 0.477


,Train,Test
Accuracy,0.8068,0.8006
AUC,0.8741,0.8407
Precision,0.3469,0.3283
Recall,0.7849,0.7198
F1,0.4811,0.4509
MCC,0.4319,0.3895


In [14]:
# Decision Tree: tune depth, minimum leaf size and class_weight with 5-fold stratified CV (scored on F1), then compare train vs test metrics
from sklearn.tree import DecisionTreeClassifier

dt_pipe = Pipeline([("scaler", StandardScaler()),
                    ("model", DecisionTreeClassifier(random_state=42))])

dt_grid = GridSearchCV(dt_pipe,
                       param_grid={"model__max_depth": [3, 5, 8, 12, None],
                                   "model__min_samples_leaf": [1, 5, 10, 20],
                                   "model__class_weight": [None, "balanced"]},
                       scoring="f1", cv=5, return_train_score=True, n_jobs=-1)
dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_
results["Decision Tree"] = evaluate(best_dt, X_test, y_test)

print("Best params:", dt_grid.best_params_, "| CV F1:", round(dt_grid.best_score_, 4))
pd.DataFrame({"Train": evaluate(best_dt, X_train, y_train), "Test": results["Decision Tree"]}).round(4)

Best params: {'model__class_weight': 'balanced', 'model__max_depth': 12, 'model__min_samples_leaf': 1} | CV F1: 0.8206


,Train,Test
Accuracy,0.9845,0.9588
AUC,0.9987,0.9262
Precision,0.8806,0.7900
Recall,1.0000,0.8681
F1,0.9365,0.8272
MCC,0.9302,0.8050
